In [ ]:
# ─── 1. Install ───────────────────────────────────────────────────────────────
!pip uninstall -y peft accelerate transformers -q
!pip install -q \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.16.1 \
  safetensors==0.4.2 \
  scikit-learn \
  numpy==1.26.4 \
  Pillow

# ─── Restart runtime ──────────────────────────────────────────────────────────
import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 10.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not cur

In [ ]:
# ─── 2. Mount Drive ───────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/ResNet18_IFND_Final"
import os, time, json
os.makedirs(SAVE_DIR, exist_ok=True)

# ─── 3. Imports ───────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from io import BytesIO
from PIL import Image
from torchvision import models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from datasets import load_dataset, Image as HFImage
from transformers import get_linear_schedule_with_warmup

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ─── 4. Load dataset ──────────────────────────────────────────────────────────
print("⏳ Loading dataset IFND-multimodal...")
hf_dataset = load_dataset("Nhat243/IFND-multimodal")
hf_dataset = hf_dataset.cast_column("image", HFImage(decode=False))
print(hf_dataset)

# Kiểm tra label distribution
train_labels_list = hf_dataset['train']['label']
print(f"📊 Label distribution - Train: REAL={list(train_labels_list).count(1)}, FAKE={list(train_labels_list).count(0)}")

# ─── 5. Transform & Dataset Class ─────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class IFNDImageDataset(torch.utils.data.Dataset):
    def __init__(self, hf_split, transform):
        self.data = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        label = int(sample['label'])
        try:
            img_bytes = sample['image']['bytes']
            if img_bytes:
                image = Image.open(BytesIO(img_bytes)).convert("RGB")
            else:
                image = Image.new("RGB", (224, 224), (128, 128, 128))
        except Exception:
            image = Image.new("RGB", (224, 224), (128, 128, 128))
        return self.transform(image), torch.tensor(label, dtype=torch.long)

train_dataset = IFNDImageDataset(hf_dataset['train'], transform)
val_dataset = IFNDImageDataset(hf_dataset['validation'], transform)
test_dataset = IFNDImageDataset(hf_dataset['test'], transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

print(f"✅ Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ─── 6. Model ─────────────────────────────────────────────────────────────────
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 ResNet-18 on IFND")
print(f"   Total params    : {total_params/1e6:.2f}M")
print(f"   Trainable params: {trainable_params/1e6:.2f}M")
print(f"   Alignment: REAL=1, FAKE=0")

# ─── 7. Training setup (LR = 1e-5 để đồng nhất) ──────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)  # 🔥 SỬA: 2e-5 → 1e-5
total_steps = len(train_loader) * 5
warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps,
    num_training_steps=total_steps)

# ─── 8. Evaluation function ───────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)
            all_probs.extend(probs[:, 1])
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary")
    return {
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": roc_auc_score(all_labels, all_probs)
    }

# ─── 9. Training loop ─────────────────────────────────────────────────────────
print("\n🚀 Training ResNet-18 on IFND (LR=1e-5)...")
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

best_f1 = 0
patience = 0
best_path = os.path.join(SAVE_DIR, "best_model.pth")

train_start = time.time()
for epoch in range(1, 6):
    epoch_start = time.time()

    # Training
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    val_metrics = evaluate(model, val_loader)
    epoch_time = (time.time() - epoch_start) / 60

    print(f"Epoch {epoch}/5 | Loss: {train_loss:.4f} | "
          f"Acc: {val_metrics['accuracy']*100:.2f}% | "
          f"F1: {val_metrics['f1']*100:.2f}% | "
          f"AUC: {val_metrics['auc']:.4f} | "
          f"Time: {epoch_time:.1f}min")

    if val_metrics['f1'] > best_f1:
        best_f1 = val_metrics['f1']
        torch.save(model.state_dict(), best_path)
        print(f"  ✅ Best model saved (F1={best_f1*100:.2f}%)")
        patience = 0
    else:
        patience += 1
        if patience >= 2:
            print(f"  ⏹ Early stopping at epoch {epoch}")
            break

total_train_time = (time.time() - train_start) / 60  # minutes
peak_vram = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0

print(f"\n⏱️ Total training time: {total_train_time:.2f} minutes")
print(f"💾 Peak VRAM: {peak_vram:.2f} GB")

# ─── 10. Load best model & Test ───────────────────────────────────────────────
model.load_state_dict(torch.load(best_path))
print("\n📊 Evaluating on IFND TEST split...")
test_metrics = evaluate(model, test_loader)

# ─── 11. Latency Measurement ──────────────────────────────────────────────────
model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)

# Warmup
with torch.no_grad():
    for _ in range(50):
        _ = model(dummy_input)

# Measure latency
latencies = []
with torch.no_grad():
    for _ in range(200):
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = model(dummy_input)
        if device == "cuda":
            torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
latency_mean = np.mean(latencies)
latency_p50 = np.percentile(latencies, 50)
latency_p95 = np.percentile(latencies, 95)

print(f"\n⚡ Inference Latency (batch_size=1):")
print(f"   Mean: {latency_mean:.2f} ms")
print(f"   P50 : {latency_p50:.2f} ms")
print(f"   P95 : {latency_p95:.2f} ms")

# ─── 12. Results ──────────────────────────────────────────────────────────────
results = {
    "Dataset": "IFND-multimodal",
    "Model": "ResNet-18",
    "Method": "Full Fine-Tuning",
    "Label_Mapping": "1=REAL, 0=FAKE",
    "Test_Results": {
        "Accuracy (%)": round(test_metrics['accuracy'] * 100, 2),
        "Precision (%)": round(test_metrics['precision'] * 100, 2),
        "Recall (%)": round(test_metrics['recall'] * 100, 2),
        "F1 (%)": round(test_metrics['f1'] * 100, 2),
        "AUC": round(test_metrics['auc'], 4)
    },
    "Latency_ms": {
        "Mean": round(latency_mean, 2),
        "P50": round(latency_p50, 2),
        "P95": round(latency_p95, 2)
    },
    "Hardware_Stats": {
        "Total_Params_M": round(total_params / 1e6, 2),
        "Trainable_Params_M": round(trainable_params / 1e6, 2),
        "Training_Time_Min": round(total_train_time, 2),
        "Peak_VRAM_GB": round(peak_vram, 2),
        "Learning_Rate": 1e-5,
        "Batch_Size": 32,
        "Epochs": 5,
        "Input_Size": "224x224"
    }
}

# Hiển thị kết quả
print("\n" + "="*60)
print("📊 KẾT QUẢ RESNET-18 TRÊN IFND")
print("="*60)
print(f"📍 Test Set: IFND-multimodal")
print(f"   Accuracy : {results['Test_Results']['Accuracy (%)']}%")
print(f"   Precision: {results['Test_Results']['Precision (%)']}%")
print(f"   Recall   : {results['Test_Results']['Recall (%)']}%")
print(f"   F1 Score : {results['Test_Results']['F1 (%)']}%")
print(f"   AUC      : {results['Test_Results']['AUC']}")
print(f"\n⚡ Performance:")
print(f"   Latency (P50): {latency_p50:.2f} ms/sample")
print(f"   Training Time: {total_train_time:.2f} min")
print(f"   Peak VRAM    : {peak_vram:.2f} GB")
print("="*60)

# Lưu JSON
with open(os.path.join(SAVE_DIR, "results_IFND_ResNet18.json"), "w") as f:
    json.dump(results, f, indent=4)

# Lưu CSV
df_results = pd.DataFrame([results["Test_Results"]])
df_results.to_csv(os.path.join(SAVE_DIR, "results_IFND_ResNet18.csv"))

# Lưu model
torch.save(model.state_dict(), os.path.join(SAVE_DIR, "final_model.pth"))

print(f"\n✅ Results saved to: {SAVE_DIR}")
print(f"   - results_IFND_ResNet18.json")
print(f"   - results_IFND_ResNet18.csv")
print(f"   - final_model.pth")

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda
⏳ Loading dataset IFND-multimodal...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/8416 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1052 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1053 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 8416
    })
    validation: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 1052
    })
    test: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 1053
    })
})
📊 Label distribution - Train: REAL=6447, FAKE=1969
✅ Train: 8416 | Val: 1052 | Test: 1053
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 130MB/s]



📊 ResNet-18 on IFND
   Total params    : 11.18M
   Trainable params: 11.18M
   Alignment: REAL=1, FAKE=0

🚀 Training ResNet-18 on IFND (LR=1e-5)...
Epoch 1/5 | Loss: 0.6059 | Acc: 79.47% | F1: 88.05% | AUC: 0.6601 | Time: 1.9min
  ✅ Best model saved (F1=88.05%)
Epoch 2/5 | Loss: 0.4530 | Acc: 78.99% | F1: 87.54% | AUC: 0.7084 | Time: 1.9min
Epoch 3/5 | Loss: 0.4011 | Acc: 78.61% | F1: 87.27% | AUC: 0.7258 | Time: 1.8min
  ⏹ Early stopping at epoch 3

⏱️ Total training time: 5.61 minutes
💾 Peak VRAM: 0.88 GB

📊 Evaluating on IFND TEST split...

⚡ Inference Latency (batch_size=1):
   Mean: 2.76 ms
   P50 : 2.67 ms
   P95 : 3.15 ms

📊 KẾT QUẢ RESNET-18 TRÊN IFND
📍 Test Set: IFND-multimodal
   Accuracy : 79.11%
   Precision: 79.2%
   Recall   : 98.64%
   F1 Score : 87.86%
   AUC      : 0.6388

⚡ Performance:
   Latency (P50): 2.67 ms/sample
   Training Time: 5.61 min
   Peak VRAM    : 0.88 GB

✅ Results saved to: /content/drive/MyDrive/ResNet18_IFND_Final
   - results_IFND_ResNet18.json
 

In [ ]:
print("⏳ Đang ngắt kết nối phiên làm việc để giải phóng GPU...")
from google.colab import runtime
time.sleep(10) # Đợi đồng bộ Drive
runtime.unassign()

⏳ Đang ngắt kết nối phiên làm việc để giải phóng GPU...
